# Rule-Based Control Baseline

This notebook runs CityLearn's `BasicRBC` controller for one complete episode. The controller follows a fixed hourly schedule and does not learn from rewards. Its results provide a reference for later SAC and MARLISA experiments.

## BasicRBC rules

| Time | DHW and electrical storage | Cooling device |
| --- | ---: | ---: |
| 22:00–08:00 | `+0.091` (charge) | `0.4` |
| 09:00–21:00 | `-0.08` (discharge) | `0.8` |

Storage and device constraints can reduce the action that is physically carried out.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from run_rbc import run_basic_rbc

## Run one episode

The environment uses `central_agent=False`, giving each building a separate action vector. Results are saved under `outputs/rbc/`.

In [ ]:
result = run_basic_rbc(
    schema=PROJECT_ROOT / "notebooks" / "data" / "schema.json",
    output_directory=PROJECT_ROOT / "outputs" / "rbc",
    random_seed=42,
)

control_steps = result["actions"]["time_step"].nunique()
print(f"Completed {control_steps} control steps.")
print(f"Results saved to: {result['output_directory']}")

## Inspect the actions

The following table and plot show the requested actions during the first 24 time steps.

In [ ]:
actions = result["actions"]
first_day = actions.loc[actions["time_step"] < 24]

display(first_day.head(12))

figure, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
for axis, (building, data) in zip(axes, first_day.groupby("building")):
    for action_name, action_data in data.groupby("action"):
        axis.step(
            action_data["time_step"],
            action_data["value"],
            where="post",
            label=action_name,
        )
    axis.axhline(0.0, color="black", linewidth=0.8)
    axis.set_title(building)
    axis.set_ylabel("Action")
    axis.legend(loc="upper right")

axes[-1].set_xlabel("Time step")
figure.suptitle("BasicRBC requested actions: first 24 time steps")
figure.tight_layout()
plt.show()

## Evaluate the controller

For the ratio-based KPIs, `1.0` means equal to CityLearn's built-in baseline, values below `1.0` are improvements, and values above `1.0` are worse. Discomfort values are direct measurements and should be interpreted separately.

In [ ]:
kpi_table = result["kpi_table"].round(3)
display(kpi_table.dropna(how="all"))

In [ ]:
selected_metrics = [
    "carbon_emissions_total",
    "cost_total",
    "electricity_consumption_total",
    "daily_peak_average",
    "ramping_average",
    "discomfort_proportion",
]

available_metrics = [metric for metric in selected_metrics if metric in kpi_table.index]
display(kpi_table.loc[available_metrics])